Imports

In [ ]:
import wrds
import pandas as pd
import numpy as np
import cvxpy as cp
from cvxpy import MOSEK
import os
from scipy.stats import f as f_dist
import warnings

Connected to WRDS


**Variables and conventions:**

**Fixed Quantities**
| Variable | Meaning | Type |
| -------- | ------- | ---- |
| $H$                           | combined factor-risk + estimation-uncertainty matrix | $m \times m$ matrix |
| $Q$                           | eigenvectors of $H$ | $m \times m$ matrix |
| $\theta$ (theta)              | eigenvalues of $H$ | $m \times 1$ vector |
| $\theta_{max}$ (theta_max)    | max eigenvalue of $H$ | scalar (float) |
| $A$                           | rotated factor exposure map ($Q^T H^{frac{1}{2}}G^{frac{1}{2}}V_0) | $m \times n$ matrix |

**Per-asset / uncertainty-set parameters**
| Variable | Meaning | Type |
| -------- | ------- | ---- |
| $\mu_0$ (mu0)                 | point estimate of mean returns | $n \times 1$ vector |
| $\alpha$ (alpha)              | lower bound on required portfolio return | scalar (float) |
| $\gamma$ (gamma)              | box uncertainty radius on mean returns | $n \times 1$ vector |
| $\rho$ (rho)                  | ellipsoidal uncertainty radius on factor loadings | $n \times 1$ vector |
| $D$                           | covariance matrix, where $d_i$ lies in interval $[\underline{d}_i, \bar{d}_i], i = 1, ... , n$ | $n \times n$ matrix |
| $\bar{D}$                     | Diagonal matrix of box uncertainty upper bound values of $d$ on idiosyncratic variance | $n \times n$ matrix |
| $\bar{d}$                     | box uncertainty upper bound on idiosyncratic variance | $n \times 1$ vector |

**Dimensions**
| Variable | Meaning |
| -------- | ------- |
| $n$      | number of assets  |
| $m$      | number of factors |

**Decision Variables**
| Variable | Meaning | Type |
| -------- | ------- | ---- |
| $\phi$ (phi)      | portfolio weights      | n x 1 vector |
| $\lambda$ (lamda) | factor variance budget | scalar (float) |
| $\delta$ (delta)  | idiosyncratic variance budget | scalar (float) |
| $\zeta$ (zeta)    | absolute value bound on phi (for C3, C4) | n x 1 vector |
| $\sigma$ (sigma)  | S-procedure multiplier | scalar (float) |
| $\tau$ (tau)            | auxiliary variable in C6/C8 | scalar (float) |
| $t$            | per-factor auxiliary variable, length m (C6/C9) | $m \times 1$ vector |






Helpers

In [ ]:
def compute_matrix_power(A: np.ndarray, n: float) -> np.ndarray:
    """
    Compute A^n, assuming A is positive semidefinite
    """
    # 1. Compute eigenvalues (w) and eigenvectors (v)
    eigenvalues, eigenvectors = np.linalg.eigh(A)
    eigenvalues = np.clip(eigenvalues, 0, None) # for numerical stability

    # 2. Raise eigenvalues to the power of n
    # (Avoid division by zero by ensuring eigenvalues are positive)
    inv_sqrt_eigenvalues = eigenvalues ** n

    # 3. Reconstruct the matrix: V * diag(λ^n) * V.T
    A_n = eigenvectors @ np.diag(inv_sqrt_eigenvalues) @ eigenvectors.T

    return A_n

def compute_fixed_quantities(G: np.ndarray, F: np.ndarray, V0: np.ndarray) -> tuple:
    """
    Compute H, its eigendecomposition, and A
    """

    #Ensure G is stable
    if np.linalg.cond(G) > 1e10:
        G = G + 1e-8 * np.eye(G.shape[0])

    G_minus_half = compute_matrix_power(G, -0.5)
    H = G_minus_half @ F @ G_minus_half

    eigenvalues, eigenvectors = np.linalg.eigh(H)
    Q = eigenvectors
    Lambda = np.diag(eigenvalues)
    H_decomposed = Q @ Lambda @ Q.T

    theta = eigenvalues
    theta_max = theta.max()

    H_sqrt = compute_matrix_power(H, 0.5)
    G_sqrt = compute_matrix_power(G, 0.5)
    A = Q.T @ H_sqrt @ G_sqrt @ V0

    """
    For testing:
    m = F.shape[0]
        n = V0.shape[1]
    
    
    if not np.allclose(H, H.T) or not A.shape == (m,n) or not np.allclose(H, H_decomposed):
                print("Computational error")
                return ()
    """
    

    return (H, Q, Lambda, theta, theta_max, A)

3. Solver

In [78]:
def solve_gi_socp(H: np.ndarray, Q: np.ndarray, theta: np.ndarray, 
                  theta_max: float, A: np.ndarray, mu0: np.ndarray, 
                  gamma: np.ndarray, rho: np.ndarray, d_bar: np.ndarray, alpha: float) -> dict:

    #Find dimensions
    n = d_bar.shape[0]
    m = H.shape[0]
    
    #defining decision and objective variables
    phi = cp.Variable(n)
    lam = cp.Variable()
    delta = cp.Variable()
    zeta  = cp.Variable(n)
    sigma = cp.Variable()
    tau = cp.Variable(nonneg=True)
    t = cp.Variable(m)
    n_ones = np.ones(n)
    m_ones = np.ones(m)

    #Construct SOC constraints
    #C1:
    sqrt_d_bar = np.sqrt(d_bar)
    v1 = cp.multiply(2*sqrt_d_bar, phi)
    soc_arg1 = cp.hstack([v1, 1 - delta])
    C1 = cp.SOC(1 + delta, soc_arg1)

    #C8:
    r = rho.T @ zeta
    soc_arg8 = cp.hstack([2 * r, sigma - tau])
    C8 = cp.SOC(sigma + tau, soc_arg8)

    #C9:
    w = A @ phi
    C9_list = [cp.SOC(1 - sigma * theta[i] + t[i], cp.hstack([2 * w[i], 1 - sigma * theta[i] - t[i]])) for i in range(m)]

    #Construct linear constraint list:
    C2 = mu0 @ phi - gamma @ phi >= alpha       #C2
    C3 = phi <= zeta                            #C3
    C4 = -phi <= zeta                           #C4
    C5 = n_ones @ phi == 1                      #C5
    C6 = tau + m_ones @ t <= lam                #C6
    C7 = sigma <= 1.0/theta_max                 #C7
    C10 = phi >= 0                              #C10

    linear_constraints = [C2, C3, C4, C5, C6, C7, C10]

    #Bundle SOC constraints with linear constraints
    all_constraints = [C1, C8] + C9_list + linear_constraints 

    #Define objective
    problem = cp.Problem(cp.Minimize(lam + delta), all_constraints)
    try:
        problem.solve(solver=cp.MOSEK)
    except (cp.error.SolverError):
        problem.solve(solver=cp.ECOS)  # Backup solver
        

    if not problem.status == "optimal":
        raise ValueError("Solver solution was not optimal, try again with alpha = 0")

    values_dict = {
        "phi": phi.value,
        "lam": lam.value,
        "delta": delta.value,
        "sigma": sigma.value,
        "zeta": zeta.value,
        "t": t.value,
        "tau": tau.value,
        "problem.value": problem.value,
        "mu1": 1.0,                         # proven by stationarity. delta has coeff + 1 in objective, -1 in C1
        "mu2": C2.dual_value,
        "mu3": C3.dual_value,
        "mu4": C4.dual_value,
        "mu5": C5.dual_value,
        "mu6": 1.0,
        "mu7": C7.dual_value,
        "mu10": C10.dual_value,
    }

    # double check mu1
    c1_tight = np.isclose((phi.value**2 * d_bar).sum(), delta.value, atol=1e-4)
    if not c1_tight:
        warnings.warn("C1 not tight — mu1=1 assumption may not hold, check formulation")

    # unpack mu8, mu9
    r_val = r.value
    values_dict["mu8"] = 1 / sigma.value   # = 1/sigma_opt, proven via tau-stationarity (C6, C8 only)
    predicted_tau = (rho @ zeta.value)**2 / sigma.value
    print("tau (solver):", tau.value, " predicted tau:", predicted_tau)  # expect same tiny order of magnitude, not exact match

    w_val = w.value
    eps = 1e-8
    near_zero = np.abs(w_val) < eps
    if near_zero.any():
        warnings.warn(f"w near zero for factor(s) {np.where(near_zero)[0]} - mu9 unreliable there")
    mu9_raw = np.array([-c.dual_value[1][0][0] for c in C9_list])
    values_dict["mu9"] = np.where(near_zero, np.nan, mu9_raw / np.where(near_zero, 1.0, w_val))
    

    return values_dict



Toy example: Simulate with small n and m

In [ ]:
n = 20     # number of stocks
m = 6      # number of factors
T = 60     # rolling window months

np.random.seed(42)

X = np.random.randn(m*5,m)        # create random matrix
cov = (X.T @ X) / (m*5)           # make cov matrix positive semidefinite      
cov = cov * 0.0005                # scale cov matrix to simulate realistic variances

mean = np.array([0.007, 0, 0, 0, 0, 0]) + np.random.uniform(-0.002, 0.002, size=m)   # create mean return vector for factors

f = np.random.multivariate_normal(mean, cov, T)     # create T x m factor return matrix

print(f.shape)
print(np.cov(f.T))

# Simulate true asset parameters to see how accurate estimator is
mu0_true = np.random.uniform(0.005, 0.015, size=(n,))
print(f"mu0_true: {mu0_true}")
V0_true = np.random.uniform(0.3, 1.5, size=(m,n))
print(f"V0_true: {V0_true}")

print(f"mu0 shape: {mu0_true.shape}")
print(f"V0 shape: {V0_true.shape}")


#simulate return panel: r_it = mu0_true_i + V0_true[:,i] * f_t + eps_it
sigma_eps = np.random.uniform(0.02, 0.05, size=(n,))
eps = np.random.normal(0,1, size=(T,n)) * sigma_eps

r = mu0_true + f @ V0_true + eps
print(f"r shape: {r.shape}")
print(r.mean(axis=0))

# OLS regression check
A_reg = np.hstack([np.ones((T,1)),f])
print(f"A_reg shape: {A_reg.shape}")


x_hat, _, _, _ = np.linalg.lstsq(A_reg, r, rcond=None)
print(f"x_hat shape: {x_hat.shape}")
mu_hat = x_hat[0, :]
V_hat = x_hat[1:, :]
print("mu_hat: ", mu_hat)
print("mu0_true:", mu0_true)
print("max abs diff (mu):", np.max(np.abs(mu_hat - mu0_true)))
print("max abs diff (V):", np.max(np.abs(V_hat - V0_true)))

#s2_i = ||r_i - A_reg @ x_hat_i ||^2 / (T-m-1)
resid = r - A_reg @ x_hat
s2 = (resid**2).sum(axis=0) / (T - m -1)
print(f"s2: {s2}, sigma_eps**2: ", sigma_eps**2)


# Computing F and G
F = np.cov(f.T)
B = f.T
ones = np.ones((T,1))
G = np.linalg.inv(B @ B.T - (1/T) * (B @ ones) @ (ones.T @ B.T))

print(f"F shape: {F.shape}")
print(f"G shape: {G.shape}")
print(np.allclose(F, F.T))
print(np.allclose(G, G.T))

H, Q, Lamda, theta, theta_max, A = compute_fixed_quantities(G, F, V0_true)
all_positive = all(x > 0 for x in theta)
print(all_positive)  # Returns: True
print(f"A shape: {A.shape}")

# Critical values and uncertainty parameters
c_full = f_dist.ppf(0.95, m+1, T-m-1)
c_mean = f_dist.ppf(0.95, 1, T-m-1)

rho = np.sqrt((m+1) * c_full * s2)
gamma = np.sqrt(np.linalg.inv(A_reg.T @ A_reg)[0][0] * c_mean * s2)
d_bar = s2

print(f"rho shape: {rho.shape}")
i = 1
for entry in rho:
    print(f"rho for stock {i}: {entry}")
    i = i + 1
print(f"gamma shape: {gamma.shape}")
print(f"d_bar shape: {d_bar.shape}")


# solve gi socp
res = solve_gi_socp(H, Q, theta, theta_max, A, mu0_true, gamma, rho, d_bar, alpha = 0.0)
for entry in res:
    print(f"{entry}: {res[entry]}")

sigma_opt = res["sigma"]
predicted_mu9 = 1 / (1 - sigma_opt * theta)
print("mu9 (from solver):", res["mu9"])
print("mu9 (predicted):  ", predicted_mu9)

# ensure weights are all positive (long-only):
print("phi.sum():", res["phi"].sum())     # should be ≈1.0
print("phi.min():", res["phi"].min())     # should now be >= -1e-6




(60, 6)
[[ 2.48440375e-04  7.82401228e-06 -8.12677340e-05 -3.91924515e-05
  -3.47526969e-05  2.16481656e-04]
 [ 7.82401228e-06  5.82498926e-04 -1.52687263e-04 -6.84278010e-05
  -2.06887803e-05 -1.12129482e-04]
 [-8.12677340e-05 -1.52687263e-04  4.63891447e-04  6.29648770e-05
   4.44861367e-05 -1.37979711e-04]
 [-3.91924515e-05 -6.84278010e-05  6.29648770e-05  2.80086283e-04
  -9.66111084e-05 -1.43384741e-04]
 [-3.47526969e-05 -2.06887803e-05  4.44861367e-05 -9.66111084e-05
   3.50210757e-04 -4.79872921e-05]
 [ 2.16481656e-04 -1.12129482e-04 -1.37979711e-04 -1.43384741e-04
  -4.79872921e-05  8.49141580e-04]]
mu0_true: [0.00971576 0.00911841 0.00848868 0.01429529 0.01330619 0.01465027
 0.00624297 0.01230867 0.0143834  0.00681233 0.00566496 0.01241121
 0.01074473 0.01341829 0.00639772 0.01295267 0.00701627 0.00663656
 0.00664266 0.01314575]
V0_true: [[1.09823666 0.92767851 0.73059658 1.35264065 0.77093413 1.27991933
  0.82696189 0.75233332 0.85521574 0.66165345 1.19713126 0.90326447
  0.5

Check if shrinking rho, gamma --> 0 is collapses to Markowitz MVO

In [80]:
# robustness check, see if reduing gamma and rho is the same as markowitz

rho_shrunk = rho * 0.001
gamma_shrunk = gamma * 0.001

res_shrunk = solve_gi_socp(H, Q, theta, theta_max, A, mu0_true, gamma_shrunk, rho_shrunk, d_bar, alpha = 0.0)
for entry in res:
    print(f"{entry}: {res_shrunk[entry]}")

# markowitz:
Sigma = V0_true.T @ F @ V0_true + np.diag(d_bar)
phi_mkw = cp.Variable(n)
prob_mkw = cp.Problem(cp.Minimize(cp.quad_form(phi_mkw, Sigma)), [np.ones(n) @ phi_mkw == 1, phi_mkw >= 0])
prob_mkw.solve()



print("max abs diff (phi_shrunk vs phi_mkw):", np.max(np.abs(res_shrunk["phi"] - phi_mkw.value)))


tau (solver): 5.698025222348895e-08  predicted tau: 1.6280220123217673e-08
phi: [1.91040437e-01 8.51265752e-03 1.35737947e-02 4.62150996e-05
 3.33369197e-05 5.56676939e-05 5.13058048e-05 3.82730352e-05
 3.10323967e-02 6.72388961e-02 6.42885770e-02 3.33412774e-05
 2.90758470e-01 2.37455607e-05 1.47848344e-02 2.31355047e-05
 5.04658578e-05 4.21664525e-05 3.18351769e-01 2.05166263e-05]
lam: 0.0005889307989610917
delta: 0.00018054217476204393
sigma: 4.869553397358686
zeta: [1.91040437e-01 1.02268009e+00 1.35737947e-02 4.62150996e-05
 3.33369197e-05 5.56676939e-05 5.13058048e-05 3.82730352e-05
 3.10323967e-02 6.72388961e-02 6.42885770e-02 3.33412774e-05
 2.90758470e-01 2.37455607e-05 1.47848344e-02 2.31355047e-05
 5.04658578e-05 4.21664525e-05 3.18351769e-01 2.05166263e-05]
t: [1.94923990e-04 1.97594900e-04 1.55982584e-04 5.96628400e-06
 1.50672154e-06 3.28993396e-05]
tau: 5.698025222348895e-08
problem.value: 0.0007694729737231356
mu1: 1.0
mu2: 0.0
mu3: [8.02211412e-09 1.34644447e-08 7.0068